In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision.datasets import FakeData
from torchvision.transforms import v2 as transforms
import torchvision.models as models

# =====================================================================
# 0. 실무 유틸리티 함수
# =====================================================================
def freeze_bn_layers(model):
    """
    미세 조정 시 BatchNorm 레이어의 running statistics(평균, 분산) 오염을 방지하기 위해
    모든 BN 층을 eval() 모드로 강제 고정합니다.
    """
    for module in model.modules():
        if isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            module.eval()


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"사용 장치: {device}")

    # =====================================================================
    # 1. 데이터셋 준비 (실행 검증용 가상 데이터셋)
    # =====================================================================
    transform = transforms.Compose([
        transforms.ToImage(),
        transforms.ToDtype(torch.float32, scale=True),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # 3개 클래스를 가진 가상 이미지를 생성하여 DataLoader 구성
    dataset = FakeData(size=128, image_size=(3, 224, 224), num_classes=3, transform=transform)
    train_loader = DataLoader(dataset, batch_size=16, shuffle=True)
    criterion = nn.CrossEntropyLoss()

    # =====================================================================
    # 2. 사전 학습 모델 로드 및 분류기(Classifier) 교체
    # =====================================================================
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

    # 새로운 분류 영역 정의 (새로 생성된 층은 기본적으로 requires_grad = True)
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.5),
        nn.Linear(num_features, 256),
        nn.ReLU(),
        nn.Linear(256, 3) # 최종 출력 클래스: 3
    )
    model.to(device)

    # =====================================================================
    # [Phase 1] 특징 추출 (Feature Extraction)
    # =====================================================================
    print("\n==================================================")
    print(" >>> [Phase 1] 특징 추출 (Feature Extraction) 시작")
    print("==================================================")

    # 1. 뼈대 모델 파라미터 동결 (분류기 영역 제외)
    for param in model.parameters():
        param.requires_grad = False
    for param in model.fc.parameters():
        param.requires_grad = True

    # 2. 영역별 모드 설정 (Base Model: eval, Classifier: train)
    model.eval()
    model.fc.train()

    # 3. 옵티마이저 설정 (분류기 파라미터만 전달)
    optimizer_p1 = optim.Adam(model.fc.parameters(), lr=1e-3)

    # 4. Phase 1 학습 실행 (1 Epoch)
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer_p1.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_p1.step()

    print(f" - Phase 1 Loss : {loss.item():.4f}")
    print(f" - Conv1 (베이스 모델) Grad 계산 여부 : {model.conv1.weight.grad is not None} (False/None 이어야 함)")
    print(f" - FC (새로운 분류기) Grad 계산 여부  : {model.fc[1].weight.grad is not None} (True 이어야 함)")

    # =====================================================================
    # [Phase 2] 미세 조정 (Fine-Tuning)
    # =====================================================================
    print("\n==================================================")
    print(" >>> [Phase 2] 미세 조정 (Fine-Tuning) 시작")
    print("==================================================")

    # 1. 특정 상위 레이어(layer4) 및 분류기 동결 해제
    for param in model.layer4.parameters():
        param.requires_grad = True

    # 2. 차별적 학습률(Differential Learning Rate) 옵티마이저 설정
    optimizer_p2 = optim.Adam([
        {'params': model.layer4.parameters(), 'lr': 1e-5}, # 베이스 모델 상위층 (매우 낮은 학습률)
        {'params': model.fc.parameters(),     'lr': 1e-4}  # 분류기 영역 (상대적으로 높지만 하향 조정된 학습률)
    ])

    # 3. 모드 설정: 전체 train() 전환 후 BN 레이어만 eval() 모드로 고정
    model.train()
    freeze_bn_layers(model)

    # 4. Phase 2 학습 실행 (1 Epoch)
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer_p2.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_p2.step()

    print(f" - Phase 2 Loss : {loss.item():.4f}")
    print(f" - Conv1 (하위 동결 영역) Grad 계산 여부 : {model.conv1.weight.grad is not None} (False/None 이어야 함)")
    print(f" - Layer4 (상위 해제 영역) Grad 계산 여부: {model.layer4[0].conv1.weight.grad is not None} (True 이어야 함)")
    print(f" - FC (분류기 영역) Grad 계산 여부      : {model.fc[1].weight.grad is not None} (True 이어야 함)")
    print("==================================================")

if __name__ == "__main__":
    main()

사용 장치: cpu

 >>> [Phase 1] 특징 추출 (Feature Extraction) 시작
 - Phase 1 Loss : 1.1410
 - Conv1 (베이스 모델) Grad 계산 여부 : False (False/None 이어야 함)
 - FC (새로운 분류기) Grad 계산 여부  : True (True 이어야 함)

 >>> [Phase 2] 미세 조정 (Fine-Tuning) 시작
 - Phase 2 Loss : 1.1640
 - Conv1 (하위 동결 영역) Grad 계산 여부 : False (False/None 이어야 함)
 - Layer4 (상위 해제 영역) Grad 계산 여부: True (True 이어야 함)
 - FC (분류기 영역) Grad 계산 여부      : True (True 이어야 함)
